# Power Analysis — ARLMP Benchmark

A priori power analysis. Commit this notebook **before** any API calls.

**Study design:** Fully paired within-item (each URL evaluated under every format × model combination)  
**Primary test:** Friedman test for overall format effect on token count  
**Follow-up:** Wilcoxon signed-rank for 21 pairwise format comparisons  
**Correction:** Holm-Bonferroni, family-wise alpha = 0.05  
**Target:** Detect ≥10% relative difference between adjacent compact formats  
**Target power:** ≥0.90

### Key methodological notes

1. **Within-item correlation is modeled explicitly.** Because the same URL is serialized in all 7 formats, token counts across formats are correlated. Ignoring this inflates power estimates. We model correlation using a multivariate normal with a shared within-URL random effect.

2. **Friedman test is simulated directly** (not approximated from Wilcoxon power), because it is the primary omnibus test.

3. **Wilcoxon signed-rank with Holm-Bonferroni** is used for pairwise follow-ups. Correction is applied across all 21 comparisons simultaneously using the Holm step-down procedure, not a fixed Bonferroni bound.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'scipy', 'matplotlib', 'numpy', '-q'])

import numpy as np
from scipy import stats
from scipy.stats import friedmanchisquare
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Libraries loaded.')

## 1. Parameters

In [ ]:
FORMAT_NAMES = ['JSON', 'minJSON', 'YAML', 'TOML', 'TOON', 'Markdown', 'ARLMP-Min']
MEAN_TOKENS  = np.array([832, 701, 655, 630, 588, 512, 398], dtype=float)
STD_TOKENS   = np.array([120, 100,  95,  90,  85,  75,  60], dtype=float)

N_FORMATS        = len(FORMAT_NAMES)          # 7
N_COMPARISONS    = N_FORMATS * (N_FORMATS - 1) // 2  # 21
ALPHA_FAMILYWISE = 0.05
TARGET_POWER     = 0.90
N_SIM            = 5000

# Within-item (within-URL) correlation across formats.
# The same URL produces correlated token counts across formats because
# the underlying metadata object is shared. We assume a moderate
# intraclass correlation of 0.60 (conservative estimate: formats differ
# substantially in structure, so correlation is not near 1.0).
ICC = 0.60

print(f'Formats         : {N_FORMATS}')
print(f'Pairwise tests  : {N_COMPARISONS}')
print(f'Family-wise α   : {ALPHA_FAMILYWISE}')
print(f'Within-URL ICC  : {ICC}')
print(f'Simulations     : {N_SIM}')

# Hardest adjacent pair
idx_a = FORMAT_NAMES.index('TOON')
idx_b = FORMAT_NAMES.index('Markdown')
print(f'\nHardest pair    : {FORMAT_NAMES[idx_a]} ({MEAN_TOKENS[idx_a]:.0f}) vs '
      f'{FORMAT_NAMES[idx_b]} ({MEAN_TOKENS[idx_b]:.0f})')
print(f'Absolute diff   : {MEAN_TOKENS[idx_a]-MEAN_TOKENS[idx_b]:.0f} tokens')
print(f'Relative diff   : {(MEAN_TOKENS[idx_a]-MEAN_TOKENS[idx_b])/MEAN_TOKENS[idx_a]*100:.1f}%')

## 2. Correlated Data Generator

Each simulated item (URL) is drawn from a multivariate normal with:
- Marginal means = `MEAN_TOKENS` per format
- Marginal SDs = `STD_TOKENS` per format
- Covariance structure: compound symmetry with ICC = 0.60
  (i.e., Cov[format_i, format_j] = ICC × SD_i × SD_j for i ≠ j)

This models the fact that a URL with a long metadata object will tend
to produce more tokens across *all* formats (positive within-URL correlation).

In [ ]:
def build_cov_matrix(stds: np.ndarray, icc: float) -> np.ndarray:
    """
    Build compound-symmetry covariance matrix.
    Diagonal: var_i = std_i^2
    Off-diagonal: cov_ij = icc * std_i * std_j
    """
    k = len(stds)
    cov = np.outer(stds, stds) * icc
    np.fill_diagonal(cov, stds ** 2)
    return cov


def simulate_paired_data(n: int, rng: np.random.Generator) -> np.ndarray:
    """
    Simulate n items × 7 formats token count matrix.
    Returns array of shape (n, 7).
    """
    cov = build_cov_matrix(STD_TOKENS, ICC)
    data = rng.multivariate_normal(MEAN_TOKENS, cov, size=n)
    return np.maximum(data, 1.0)  # token counts must be positive


# Verify covariance structure
cov_check = build_cov_matrix(STD_TOKENS, ICC)
print('Covariance matrix (first 3x3 block):')
print(np.round(cov_check[:3, :3], 1))
print(f'\nCorrelation TOON-Markdown: '
      f'{cov_check[idx_a, idx_b] / (STD_TOKENS[idx_a] * STD_TOKENS[idx_b]):.3f}')

## 3. Power for Friedman Test (Primary Omnibus)

Friedman power is estimated by simulation: generate n correlated items,
run the Friedman test, record whether p < α. Repeat N_SIM times.

In [ ]:
def simulate_friedman_power(n: int, alpha: float = ALPHA_FAMILYWISE,
                             n_sim: int = N_SIM) -> float:
    """Simulate power for Friedman test on k=7 correlated format columns."""
    rng = np.random.default_rng(42)
    rejections = 0
    for _ in range(n_sim):
        data = simulate_paired_data(n, rng)   # shape (n, 7)
        # friedmanchisquare expects 7 separate column arrays
        _, p = friedmanchisquare(*[data[:, j] for j in range(N_FORMATS)])
        if p < alpha:
            rejections += 1
    return rejections / n_sim


sample_sizes = [200, 300, 400, 500, 750, 1000, 1250, 1500]
friedman_powers = []

print('Friedman test power (7 formats, correlated within-URL):')
print(f'{"n":>6}   {"power":>6}   pass?')
print('-' * 25)

min_n_friedman = None
for n in sample_sizes:
    power = simulate_friedman_power(n)
    friedman_powers.append(power)
    ok = '✓' if power >= TARGET_POWER else '✗'
    print(f'n={n:5d}   {power:.3f}   {ok}')
    if power >= TARGET_POWER and min_n_friedman is None:
        min_n_friedman = n

## 4. Power for Wilcoxon + Holm-Bonferroni (Pairwise Follow-up)

For each simulated dataset, all 21 pairwise Wilcoxon tests are run
simultaneously. Holm-Bonferroni correction is applied to the full
set of 21 p-values. Power is the proportion of simulations where
the hardest adjacent pair (TOON vs Markdown) is correctly rejected
after correction.

In [ ]:
from itertools import combinations

PAIRS = list(combinations(range(N_FORMATS), 2))  # 21 pairs
TARGET_PAIR = (idx_a, idx_b)   # TOON vs Markdown


def holm_bonferroni(p_values: list[float], alpha: float) -> list[bool]:
    """
    Holm-Bonferroni step-down correction.
    Returns list of booleans: True = reject H0 at family-wise alpha.
    """
    m = len(p_values)
    indexed = sorted(enumerate(p_values), key=lambda x: x[1])
    rejected = [False] * m
    for rank, (orig_idx, p) in enumerate(indexed):
        threshold = alpha / (m - rank)
        if p <= threshold:
            rejected[orig_idx] = True
        else:
            break   # Holm stops at first non-rejection
    return rejected


def simulate_wilcoxon_holm_power(n: int, n_sim: int = N_SIM) -> float:
    """
    Simulate power for the hardest pair (TOON vs Markdown) after
    Holm-Bonferroni correction across all 21 pairwise Wilcoxon tests.
    """
    rng = np.random.default_rng(42)
    rejections = 0
    target_idx = PAIRS.index(TARGET_PAIR)

    for _ in range(n_sim):
        data = simulate_paired_data(n, rng)   # (n, 7)
        p_values = []
        for i, j in PAIRS:
            _, p = stats.wilcoxon(data[:, i], data[:, j],
                                  alternative='two-sided')
            p_values.append(p)
        rejected = holm_bonferroni(p_values, ALPHA_FAMILYWISE)
        if rejected[target_idx]:
            rejections += 1
    return rejections / n_sim


wilcoxon_powers = []

print('Wilcoxon + Holm-Bonferroni power (hardest pair: TOON vs Markdown):')
print(f'{"n":>6}   {"power":>6}   pass?')
print('-' * 25)

min_n_wilcoxon = None
for n in sample_sizes:
    power = simulate_wilcoxon_holm_power(n)
    wilcoxon_powers.append(power)
    ok = '✓' if power >= TARGET_POWER else '✗'
    print(f'n={n:5d}   {power:.3f}   {ok}')
    if power >= TARGET_POWER and min_n_wilcoxon is None:
        min_n_wilcoxon = n

## 5. McNemar Power (Accuracy Outcomes)

In [ ]:
def simulate_mcnemar_power(acc_a: float, acc_b: float, n: int,
                            alpha: float, n_sim: int = N_SIM) -> float:
    """
    McNemar power for paired accuracy comparison.
    acc_a, acc_b: expected accuracy for formats A and B.
    """
    rng = np.random.default_rng(42)
    rejections = 0
    # Discordant cell probabilities
    p_ab = max(acc_a - acc_b, 0.001)   # correct under A, wrong under B
    p_ba = max(acc_b - acc_a, 0.001)   # correct under B, wrong under A
    p_both = acc_a * acc_b
    p_neither = max(1 - p_both - p_ab - p_ba, 0.001)
    probs = np.array([p_both, p_ab, p_ba, p_neither])
    probs /= probs.sum()

    for _ in range(n_sim):
        counts = rng.multinomial(n, probs)
        b, c = counts[1], counts[2]   # discordant cells
        if b + c == 0:
            continue
        chi2 = (abs(b - c) - 1) ** 2 / (b + c)
        p = 1 - stats.chi2.cdf(chi2, df=1)
        if p < alpha:
            rejections += 1
    return rejections / n_sim


# Holm-corrected alpha for 21 comparisons (most conservative step)
alpha_holm_min = ALPHA_FAMILYWISE / N_COMPARISONS   # 0.00238

print('McNemar power — safety triage: JSON(0.942) vs ARLMP-Min(0.902)')
print(f'(Using conservative Holm alpha = {alpha_holm_min:.5f})')
print(f'{"n":>6}   {"power":>6}   pass?')
print('-' * 25)
for n in [500, 750, 1000, 1250]:
    power = simulate_mcnemar_power(0.942, 0.902, n=n, alpha=alpha_holm_min)
    ok = '✓' if power >= TARGET_POWER else '✗'
    print(f'n={n:5d}   {power:.3f}   {ok}')

## 6. Power Curve Plot

In [ ]:
import os
os.makedirs('../codebook', exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sample_sizes, friedman_powers, 'ko-', linewidth=1.5,
        markersize=6, label='Friedman (omnibus)')
ax.plot(sample_sizes, wilcoxon_powers, 'ks--', linewidth=1.5,
        markersize=6, label='Wilcoxon + Holm (hardest pair)')
ax.axhline(TARGET_POWER, color='gray', linestyle=':', linewidth=1,
           label=f'Target power = {TARGET_POWER}')
ax.axvline(1000, color='black', linestyle=':', linewidth=1,
           label='Planned n = 1,000')
ax.set_xlabel('Sample size (n links)')
ax.set_ylabel('Estimated power')
ax.set_title('Power curves — correlated within-URL simulation\n'
             '(ICC = 0.60, 7 formats, Holm-Bonferroni correction)')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../codebook/power_curve.png', dpi=150)
plt.show()
print('Saved to codebook/power_curve.png')

## 7. Conclusion

In [ ]:
power_friedman_1000  = friedman_powers[sample_sizes.index(1000)]
power_wilcoxon_1000  = wilcoxon_powers[sample_sizes.index(1000)]

print('=' * 60)
print('POWER ANALYSIS CONCLUSION')
print('=' * 60)
print(f'Simulation model    : Multivariate normal, ICC = {ICC}')
print(f'Hardest comparison  : TOON vs Markdown (token count)')
print(f'Effect (absolute)   : {MEAN_TOKENS[idx_a]-MEAN_TOKENS[idx_b]:.0f} tokens')
print(f'Effect (relative)   : '
      f'{(MEAN_TOKENS[idx_a]-MEAN_TOKENS[idx_b])/MEAN_TOKENS[idx_a]*100:.1f}%')
print(f'Correction          : Holm-Bonferroni, {N_COMPARISONS} comparisons')
print(f'Target power        : {TARGET_POWER}')
print()
print(f'Friedman (omnibus)  : power = {power_friedman_1000:.3f} at n=1,000  '
      f'{"✓" if power_friedman_1000 >= TARGET_POWER else "✗"}')
print(f'Wilcoxon + Holm     : power = {power_wilcoxon_1000:.3f} at n=1,000  '
      f'{"✓" if power_wilcoxon_1000 >= TARGET_POWER else "✗"}')
print()
print(f'Min n (Friedman)    : {min_n_friedman}')
print(f'Min n (Wilcoxon)    : {min_n_wilcoxon}')
print()
print(f'Planned sample      : n=1,000 (+200 buffer → sample 1,200)')
if (power_friedman_1000 >= TARGET_POWER and
        power_wilcoxon_1000 >= TARGET_POWER):
    print(f'\n✓ n=1,000 is well-justified for both primary tests.')
else:
    print(f'\n✗ Consider increasing n.')
print('=' * 60)